In [3]:
# In this notebook, we will see what will happen if we have 30% of data per enginge
# Importing libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import pathlib as pl
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from xgboost import XGBRegressor
from pathlib import Path
# Informative sensors
informative_sensors = [
    'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7',
    'sensor_9', 'sensor_11', 'sensor_12', 'sensor_14',
    'sensor_15', 'sensor_20', 'sensor_21'
]
# Column names
column_names = ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3'] + [f'sensor_{i}' for i in range(1, 22)]
# Load the dataset
Data_raw = Path('..')/ 'data'/ 'raw'
train_df = pd.read_csv(Data_raw / 'train_FD001.txt', sep='\s+', header=None, names=column_names)
test_df = pd.read_csv(Data_raw / 'test_FD001.txt', sep='\s+', header=None, names=column_names)
rul_df = pd.read_csv(Data_raw / 'RUL_FD001.txt', sep='\s+', header=None, names=['RUL'])
# RUL construction
train_df['RUL'] = train_df.groupby('engine_id')['cycle'].transform(max) - train_df['cycle']
train_df['RUL'] = train_df['RUL'].clip(upper=125)
# Train-test split
train_engines = train_df['engine_id'].unique()
split_idx = int(len(train_engines) * 0.8)
train = train_df[train_df['engine_id'].isin(train_engines[:split_idx])]
val = train_df[train_df['engine_id'].isin(train_engines[split_idx:])]
# Normalization
scaler = StandardScaler()
train_scaled = train.copy()
train_scaled[informative_sensors] = scaler.fit_transform(train[informative_sensors])
val_scaled = val.copy()
val_scaled[informative_sensors] = scaler.transform(val[informative_sensors])
# Limited data per engine
def limited_data(df, fraction=0.3):
    limited =[]
    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id]
        n_cycles = int(len(engine_data) * fraction)
        limited.append(engine_data.head(n_cycles))
    return pd.concat(limited, ignore_index=True)
train_limited = limited_data(train_scaled, fraction=0.3)
# Sequence creation for LSTM
def create_sequences(df, sequence_length=30):
    X, y = [], []
    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id]
        for i in range(len(engine_data) - sequence_length):
            X.append(engine_data[informative_sensors].iloc[i:i+sequence_length].values)
            y.append(engine_data['RUL'].iloc[i+sequence_length])
    return np.array(X), np.array(y)


X_train_seq, y_train_seq = create_sequences(train_limited)
X_val_seq, y_val_seq = create_sequences(val_scaled)

# LSTM
model = Sequential([
    LSTM(64, input_shape=(30, 11)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')

history = model.fit(X_train_seq, y_train_seq,
                    validation_data=(X_val_seq, y_val_seq),
                    epochs=20, batch_size=64)

y_pred = model.predict(X_val_seq).flatten()
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_val_seq, y_pred)
r2 = r2_score(y_val_seq, y_pred)
print(f'Limited LSTM RMSE: {rmse:.2f}, R2: {r2:.2f}')


Epoch 1/20


d:\cmapss-predictive-maintenance\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - loss: 14446.3184 - val_loss: 7496.5273
Epoch 2/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 11555.1006 - val_loss: 5581.8882
Epoch 3/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - loss: 8260.0088 - val_loss: 3658.0066
Epoch 4/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - loss: 5104.0820 - val_loss: 2161.0234
Epoch 5/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 2616.8889 - val_loss: 1308.2329
Epoch 6/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 1095.3442 - val_loss: 1057.1012
Epoch 7/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 372.1962 - val_loss: 1137.1176
Epoch 8/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 144.8789 - val_loss: 1278.0376
Epoch 9/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 82.6564 - val_loss: 1363.6052
Epoch 10/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 74.3107 - val_loss: 1401.3641
Epoch 11/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - loss: 70.5549 - val_loss: 1412.4878
Epoch 12/20
38